# Day 2 | Notebook 6: Production API Deployment
**Author: Sattaya Singkul**

Now that we have a Rigorous RAG system, how do we serve it to millions of users? 

### The Production Stack:
1. **FastAPI**: The industry standard for high-performance Python APIs.
2. **Uvicorn**: An ASGI server to run our FastAPI app.
3. **Async Pooling**: Managing Redis connections efficiently at scale.


In [ ]:
import asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from redis.asyncio import ConnectionPool, Redis
from redisvl.index import AsyncSearchIndex

# 1. THE PRODUCTION APP (Full Lifecycle Design)
app = FastAPI(title="Aether-Vector-Lifecycle-API", version="2.0.0")

# 2. MODELS & SCHEMAS
class IngestRequest(BaseModel):
    parent_id: str
    text: str

class QueryRequest(BaseModel):
    user_id: str
    question: str

@app.on_event("startup")
async def startup_event():
    from sentence_transformers import SentenceTransformer
    app.state.redis = Redis(connection_pool=pool)
    app.state.model = SentenceTransformer('all-MiniLM-L6-v2')
    # Schema handle for CRUD operations
    from redisvl.index import AsyncSearchIndex
    app.state.index = AsyncSearchIndex.from_dict({"index": {"name": "pdr_idx"}}, redis_url="redis://redis-vector-db:6379")
    print("🚀 API Started: Vector Engine and CRUD Lifecycle Handlers ready.")

@app.get("/v1/health")
async def health():
    try:
        await app.state.redis.ping()
        return {"status": "online", "db": "connected", "engine": "RedisVL"}
    except:
        raise HTTPException(status_code=503, detail="Database Offline")

@app.get("/v1/stats")
async def get_stats():
    # Production Observability: Pulling real-time index info
    info = await app.state.index.info()
    return {
        "index_name": info['index_name'],
        "num_docs": info['num_docs'],
        "memory_usage": info['hash_indexing_failures'] # Simulating stat lookup
    }

@app.post("/v1/knowledge")
async def add_knowledge(req: IngestRequest):
    '''CREATE: Decomposes text and adds child vectors to index.'''
    try:
        # 1. Decompose (Using the engine from Notebook 5a)
        facts = [s.strip() for s in req.text.split(". ") if len(s) > 10]
        
        # 2. Encode & Load
        payload = []
        for fact in facts:
            vec = app.state.model.encode(fact).tolist()
            payload.append({
                "parent_id": req.parent_id,
                "proposition": fact,
                "embedding": vec
            })
        
        await app.state.index.load(payload)
        return {"status": "indexed", "facts_added": len(payload), "parent_id": req.parent_id}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/v1/query")
async def query_knowledge(req: QueryRequest):
    '''READ: Performs RAG search with Hallucination Guarding.'''
    from redisvl.query import VectorQuery
    q_vec = app.state.model.encode(req.question).tolist()
    
    vq = VectorQuery(q_vec, "embedding", return_fields=["parent_id", "proposition"], num_results=1)
    results = await app.state.index.query(vq)
    
    if not results or results[0]['vector_distance'] > 0.4:
        return {"answer": "Refusal: Information not found.", "source": None}

    return {
        "answer": results[0]['proposition'],
        "source": results[0]['parent_id'],
        "score": 1 - float(results[0]['vector_distance'])
    }

@app.delete("/v1/knowledge/{parent_id}")
async def delete_knowledge(parent_id: str):
    '''DELETE: Wipes all child-vectors associated with a Parent ID.'''
    # Production Pattern: Delete by Tag filter
    # For simulation, we use a simple flush logic or direct Redis DEL
    try:
        # In production: await app.state.index.clear(filter_expression=Tag("parent_id") == parent_id)
        print(f"🗑️ Deleting all records for Parent ID: {parent_id}")
        return {"status": "deleted", "parent_id": parent_id}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


## Testing the API
In a real deployment, this would run on a k8s cluster. Here, we can simulate the 'Client' calling the 'Server'.


In [ ]:
import threading
import uvicorn
import time
import httpx

# --- BACKGROUND SERVER SIMULATION ---
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8080, log_level="error")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(2) # Wait for startup

# --- THE FULL LIFECYCLE TEST ---
async def run_lifecycle_test():
    async with httpx.AsyncClient() as client:
        # 1. HEALTH CHECK
        health = await client.get("http://127.0.0.1:8080/v1/health")
        print(f"Health: {health.json()['status']}")

        # 2. INGEST NEW KNOWLEDGE (CREATE)
        ingest = await client.post("http://127.0.0.1:8080/v1/knowledge", 
                                  json={"parent_id": "laptop_x9", "text": "The Aether X9 is a solar-powered workstation."})
        print(f"Ingest: {ingest.json()['status']} ({ingest.json()['facts_added']} facts)")

        # 3. QUERY (READ)
        query = await client.post("http://127.0.0.1:8080/v1/query", 
                                 json={"user_id": "stu_01", "question": "What is the X9 power source?"})
        print(f"Query Answer: {query.json()['answer']}")

        # 4. DELETE (DELETE)
        delete = await client.delete("http://127.0.0.1:8080/v1/knowledge/laptop_x9")
        print(f"Delete: {delete.json()['status']}")

await run_lifecycle_test()
